In [ ]:
#load apikeys from .env

from dotenv import load_dotenv
import os
import getpass

load_dotenv()


if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Introduce tu  AWS Key: ")

In [ ]:
#model agnostic function for llm inference with init_chat_model and invoke()

from langchain.chat_models import init_chat_model

def generate_example_orders(model_name, prompt, provider=None, token_limit=200, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        max_tokens=token_limit,
        temperature = temperature
    )
    response = model.invoke(prompt)
    return response

In [ ]:
model_name = "claude-haiku-4-5-20251001"
provider = "anthropic"
prompt = "Generate the transcription of a pizza order"


response = generate_example_orders(
    model_name,
    prompt,
    provider
)

response

In [ ]:
#Components of the AI Message

#response.content
#response.response_metadata
#response.tool_calls
#response.invalid_tool_calls
response.usage_metadata

In [ ]:
#print out all messages

import json

def print_aimessage(ai_message):
    for key, value in ai_message.model_dump().items():                #convert Ai-Message to Python dictionary and get items
        if isinstance(value, (dict, list)):                         #check if the value is a nested structure
            print(f"\n[{key}]:")                                    
            print(json.dumps(value, indent=2, ensure_ascii=False))  #converts a Python object into a JSON-formatted string. It’s commonly used to serialize data for storage or sending
        else:
            print(f"\n[{key}]: {value}")

print_aimessage(response)

In [ ]:
#start the ollama server
#uv add langchain_ollama
# utliza un modelo descargado

model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = "Generate the transcription of a pizza order"

response = generate_example_orders(
    model_name,
    prompt,
    provider
)

print_aimessage(response)

In [ ]:
import time
import pandas as pd
from pathlib import Path
from langchain.chat_models import init_chat_model

# 1. Define model configurations
models_config = [
    # Ollama Models
    {"model": "gemma3:4b", "provider": "ollama"},
    {"model": "gemma4:e4b", "provider": "ollama"},

    {"model": "llama3.1:8b", "provider": "ollama"},

    {"model": "ministral-3:3b", "provider": "ollama"},
    {"model": "mistral:7b", "provider": "ollama"},
    {"model": "mistral-nemo:12b", "provider": "ollama"},

    {"model": "qwen3.5:0.8b", "provider": "ollama"},
    {"model": "qwen3:4b-instruct", "provider": "ollama"},
    {"model": "qwen3:8b", "provider": "ollama"},
    {"model": "qwen3.5:9b", "provider": "ollama"},
    
    # Anthropic Model
    {"model": "claude-haiku-4-5-20251001", "provider": "anthropic"},
]

prompt = "Generate the transcription of a pizza order."
results = []



# 2. Invoke models and record metadata
for config in models_config:
    model_name = config["model"]
    provider = config["provider"]
    
    try:
        # Initialize LLM via LangChain init_chat_model
        llm = init_chat_model(model_name, model_provider=provider)
        
        # Track wall-clock duration
        start_time = time.perf_counter()
        response = llm.invoke(prompt)
        end_time = time.perf_counter()
        
        duration = round(end_time - start_time, 3)
        answer = response.content
        
        # Extract usage metadata
        usage_metadata = getattr(response, "usage_metadata", {}) or {}
        input_tokens = usage_metadata.get("input_tokens", None)
        output_tokens = usage_metadata.get("output_tokens", None)
        
        
        results.append({
            "modelname": model_name,
            "provider": provider,
            "duration_sec": duration,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "answer": answer
        })
        print(f"Executed {model_name} in {duration} seconds. Input tokens: {input_tokens}, Output tokens: {output_tokens}, Answer: {answer[:50]}...")  # Print first 50 chars of answer for brevity
        
    except Exception as e:
        print(f"Error executing {model_name}: {e}")

# 3. Create Pandas DataFrame
df = pd.DataFrame(results)

script_dir = Path.cwd()
data_path = script_dir.parent / "data" / "performance"

data_path.mkdir(parents=True, exist_ok=True)

df.to_csv(data_path / "performance_test.csv", index=False)

# Display DataFrame 
df

In [ ]:
# save df to csv
script_dir = Path.cwd()
data_path = script_dir.parent / "data" / "performance"

data_path.mkdir(parents=True, exist_ok=True)

df.to_csv(data_path / "performance_test.csv", index=False)

In [ ]:
df.sort_values(by="duration_sec", ascending=True, inplace=False)

In [ ]:
answer_list = df ["answer"].to_list()
for idx, answer in enumerate(answer_list):
    print(f"Answer from model {df['modelname'][idx]} Output tokens: {df['output_tokens'][idx]}:\n {answer}\n")
    print("------------------------------------------------------------\n")

In [ ]:
#to manage results more easely we only want the turn with the order.
#adapt prompt: Only generate the customers turn with the order.
model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = """
Generate a pizza order transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.
"""


response = generate_example_orders(
    model_name,
    prompt,
    provider,
    temperature
)

print(response.content)
print(len(response.content))


In [ ]:
# The standard way to ensure consistent output format from LLMs is to use Pydantic models to define and validate the expected structure, 
# which can then be enforced via the API's structured-output feature

from typing import List
from pydantic import BaseModel, Field

class Pizza(BaseModel):
    """Transcription of the pizza order."""

    pizza_order: str = Field(
        ..., description="The pizza order transcription."
    )

class Pizza_order_list(BaseModel):
    """ Generated Pizza orders"""
    # Creates a model so that we can extract multiple entities.
    order_list: List[Pizza]



In [ ]:
from langchain.chat_models import init_chat_model

def generate_structured_example_orders(model_name, prompt, provider=None, temperature = 0.7):
    model = init_chat_model(
        model_name,
        model_provider=provider,
        temperature = temperature
    )
    structured_llm = model.with_structured_output(schema=Pizza_order_list, include_raw=True)
    response = structured_llm.invoke(prompt)
    return response

In [ ]:
model_name = "qwen3:4b-instruct"
provider = "ollama"
prompt = """
Generate 5 pizza orders transcribed as if the customer was placing them verbally over the phone. 
Only generate the customers turn with the order.
"""

response = generate_structured_example_orders(
    model_name,
    prompt,
    provider,
    temperature
)
response

In [ ]:
#transform pydantic object to Python dict using model_dump()
print(response["raw"].usage_metadata)
print(response["parsed"])
response_dict = response["parsed"].model_dump()
print(type(response_dict))
response_dict

In [ ]:
#get the values of the dict 
for order in response_dict['order_list']:
    print(order ['pizza_order'])

In [ ]:
# save  response (pydantic object) as json 
from datetime import datetime
from pathlib import Path


def save_orders_to_file(content) -> Path:
    data_path = Path.cwd().parent / "data" / "example_orders"
    data_path.mkdir(parents=True, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_file = data_path / f"pedidos_pizza_{timestamp}.json"

    output_file.write_text(
        content.model_dump_json(indent=2),
        encoding="utf-8",
    )

    return output_file

In [ ]:
save_orders_to_file(response)

In [ ]:
from pathlib import Path


def load_orders_as_list(filename: str) -> list[str]:
    """Lädt eine JSON-Datei aus data/example_orders und gibt die Orders als Liste zurück."""
    data_path = Path.cwd().parent / "data" / "example_orders"
    path = data_path / filename
    data = json.loads(path.read_text(encoding="utf-8"))

    return [order["pizza_order"] for order in data["order_list"]]

    


In [ ]:
python_list = load_orders_as_list("pedidos_pizza_20260914_115625.json")
print(type(python_list))
python_list
